<div style="text-align: center;">

  <img src="img/LogoUesc.png" alt="UESC" style="width: 60px; display: block; margin: 0 auto 5px auto;">

  <div style="font-size: 20px;"><strong>UNIVERSIDADE ESTADUAL DE SANTA CRUZ - UESC</strong></div>
  <div style="font-size: 18px;">DEPARTAMENTO DE ENGENHARIAS E COMPUTAÇÃO - DEC</div>
  <div style="font-size: 18px;">Grupo de Pesquisas em Simulações e Controle de Processos</div>

  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

  <div style="font-size: 22px; font-weight: bold; margin-bottom: 10px;">PROGRAMAÇÃO EM PYTHON PARA AS ENGENHARIAS</div>

  <div style="font-size: 18px; margin-top: 10px;"><strong>Tema</strong>: Introdução a Redes Neurais Artificiais - Básico</div>
  <div style="font-size: 18px;"><strong>Prof. Dr. E.R.Edwards</strong></div>

  <div style="font-size: 16px; margin-top: 15px;">Ilhéus - BA, Abril de 2025</div>
  
  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

</div>

**Sumário**

- [1. Problema de engenharia e enunciado](#secao-1)
- [2. Dados e codificação das classes](#secao-2)
- [3. Normalização das entradas](#secao-3)
- [4. Arquitetura e funções de ativação](#secao-4)
- [5. Treinamento da rede neural](#secao-5)
- [6. Previsões e interpretação dos resultados](#secao-6)


<div id="secao-1" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  1. Problema de engenharia e enunciado
</div>


__Enunciado__:

Em uma planta de processos químicos, diferentes reatores são utilizados conforme as características da reação, como a necessidade de agitação contínua, regime de operação ou tempo de residência. Neste exercício didático, utilizaremos apenas a __massa de catalisador__ e a __temperatura__ de operação como variáveis de entrada. Na prática, a seleção do tipo de reator depende de outros fatores, como cinética, mistura, transferência de calor e regime operacional.

A tabela a seguir apresenta informações de quatro reatores distintos. O objetivo é treinar uma rede neural que consiga classificar o tipo de reator com base nessas duas variáveis operacionais.

<img src="img/Tab_08.png" alt="UESC" style="width: 650px; display: block; margin: 0 auto 5px auto;">
<p style="text-align: center;"><strong>Figura: Dados de treinamento dos quatro reatores.</strong></p>

__Objetivo__:

Desenvolver uma rede neural simples de classificação que receba como entrada a __massa de catalisador__ e a __temperatura__, e como saída o __tipo de reator__.

__Instruções__:

1. Codifique os tipos de reator com vetores binários (ex: __CSTR__ = $[1, 0, 0]$, __PFR__ = $[0, 1, 0]$, __BATCH__ = $[0, 0, 1]$)
2. Normalize os dados de entrada.
3. Implemente uma rede neural com:
   - 2 entradas (massa e temperatura)
   - 1 camada oculta com alguns neurônios (ex: 3 ou 4)
   -3 saídas (representando os três tipos de reator)
4. Treine a rede com os dados fornecidos.
5. Teste com novos dados simulados, por exemplo:

<img src="img/Tab_09.png" alt="UESC" style="width: 350px; display: block; margin: 0 auto 5px auto;">
<p style="text-align: center;"><strong>Figura: Condições operacionais propostas para teste do classificador.</strong></p>

6. Analise a previsão da rede e discuta se a classificação parece coerente com os dados de entrada.

<div id="secao-2" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  2. Dados e codificação das classes
</div>


Os quatro exemplos são representados por duas variáveis de entrada e três classes de saída. A codificação *one-hot* associa um vetor binário a cada classe. Como o conjunto é muito pequeno, os resultados terão caráter exclusivamente ilustrativo.

In [15]:
import numpy as np

# Dados de entrada: [massa (g), temperatura (°C)]
entradas = np.array([
    [250, 320],  # R-101
    [300, 350],  # R-102
    [275, 340],  # R-103
    [225, 310],  # R-104
], dtype=np.float32)

# Saída (one-hot encoding):
# CSTR = [1, 0, 0], PFR = [0, 1, 0], BATCH = [0, 0, 1]
saidas = np.array([
    [1, 0, 0],  # R-101 → CSTR
    [0, 1, 0],  # R-102 → PFR
    [1, 0, 0],  # R-103 → CSTR
    [0, 0, 1],  # R-104 → BATCH
], dtype=np.float32)


<div id="secao-3" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  3. Normalização das entradas
</div>


Aplicamos a transformação mín–máx, $x_{norm}=(x-x_{min})/(x_{max}-x_{min})$, calculando os extremos **somente a partir dos dados de treinamento**. Para novos exemplos, reutilizaremos esses mesmos extremos.

In [16]:
def normalizar(dados):
    minimo = dados.min(axis=0)
    maximo = dados.max(axis=0)
    amplitude = maximo - minimo
    if np.any(amplitude == 0):
        raise ValueError("Uma das variáveis não apresenta variação nos dados.")
    return (dados - minimo) / amplitude, minimo, maximo

entradas_norm, entradas_min, entradas_max = normalizar(entradas)
print("Entradas normalizadas:\n", entradas_norm)


Entradas normalizadas:
 [[0.33333334 0.25      ]
 [1.         1.        ]
 [0.6666667  0.75      ]
 [0.         0.        ]]


<div id="secao-4" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  4. Arquitetura e funções de ativação
</div>


Utilizamos duas entradas, quatro neurônios ocultos com ativação ReLU e três saídas com *softmax*. As saídas da *softmax* somam 1 e são interpretadas como escores normalizados entre as três classes; com apenas quatro amostras, não devem ser consideradas probabilidades calibradas.

In [17]:
def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # estabilidade numérica
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


In [18]:
np.random.seed(0)

# Estrutura: 2 entradas, 4 neurônios ocultos, 3 saídas
pesos_entrada_oculta = np.random.randn(2, 4)
bias_oculta = np.zeros((1, 4))

pesos_oculta_saida = np.random.randn(4, 3)
bias_saida = np.zeros((1, 3))

taxa_aprendizado = 0.1
epochs = 1000


<div id="secao-5" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  5. Treinamento da rede neural
</div>


O treinamento minimiza a **entropia cruzada categórica**. Para *softmax* combinada com essa função de perda, o gradiente médio em relação aos logits é $(\hat{Y}-Y)/N$. A atualização dos parâmetros ocorre no sentido oposto ao gradiente. O erro quadrático médio não será usado como medida de treinamento, evitando misturar duas funções de perda diferentes.

In [19]:
n_amostras = len(entradas_norm)
historico_perda = []

for epoca in range(epochs):
    # Propagação direta
    z_oculta = entradas_norm @ pesos_entrada_oculta + bias_oculta
    a_oculta = relu(z_oculta)
    z_saida = a_oculta @ pesos_oculta_saida + bias_saida
    saida_predita = softmax(z_saida)

    # Entropia cruzada categórica média
    perda = -np.mean(np.sum(saidas * np.log(np.clip(saida_predita, 1e-12, 1.0)), axis=1))
    historico_perda.append(perda)

    # Retropropagação: gradiente da entropia cruzada com softmax
    d_saida = (saida_predita - saidas) / n_amostras
    d_pesos_oculta_saida = a_oculta.T @ d_saida
    d_bias_saida = np.sum(d_saida, axis=0, keepdims=True)

    d_oculta = (d_saida @ pesos_oculta_saida.T) * relu_derivada(z_oculta)
    d_pesos_entrada_oculta = entradas_norm.T @ d_oculta
    d_bias_oculta = np.sum(d_oculta, axis=0, keepdims=True)

    # Descida do gradiente
    pesos_oculta_saida -= taxa_aprendizado * d_pesos_oculta_saida
    bias_saida -= taxa_aprendizado * d_bias_saida
    pesos_entrada_oculta -= taxa_aprendizado * d_pesos_entrada_oculta
    bias_oculta -= taxa_aprendizado * d_bias_oculta

    if epoca % 200 == 0 or epoca == epochs - 1:
        print(f"Época {epoca:4d} - Entropia cruzada média: {perda:.4f}")

# Verificação do desempenho apenas nos exemplos de treinamento
z_oculta = entradas_norm @ pesos_entrada_oculta + bias_oculta
saida_treino = softmax(relu(z_oculta) @ pesos_oculta_saida + bias_saida)
classes_previstas = np.argmax(saida_treino, axis=1)
classes_reais = np.argmax(saidas, axis=1)
print(f"Acertos no treinamento: {np.sum(classes_previstas == classes_reais)}/{n_amostras}")


Época    0 - Entropia cruzada média: 1.3433
Época  200 - Entropia cruzada média: 0.3843
Época  400 - Entropia cruzada média: 0.2474
Época  600 - Entropia cruzada média: 0.1537
Época  800 - Entropia cruzada média: 0.0941
Época  999 - Entropia cruzada média: 0.0621
Acertos no treinamento: 4/4


<div id="secao-6" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  6. Previsões e interpretação dos resultados
</div>


Os novos pares de massa e temperatura são normalizados usando os limites obtidos no treinamento. A classe prevista corresponde ao maior valor da saída *softmax*. Como não há rótulos conhecidos para esses novos pontos, **não é possível avaliar a exatidão das previsões nem afirmar que houve generalização**.

In [20]:
# Novas entradas: [massa, temperatura]
novos = np.array([
    [260, 330],
    [290, 345]
], dtype=np.float32)

# Normalizar com base no conjunto original
novos_norm = (novos - entradas_min) / (entradas_max - entradas_min)

# Forward para novas previsões
z_oculta_novos = np.dot(novos_norm, pesos_entrada_oculta) + bias_oculta
a_oculta_novos = relu(z_oculta_novos)

z_saida_novos = np.dot(a_oculta_novos, pesos_oculta_saida) + bias_saida
saida_novos = softmax(z_saida_novos)

# Interpretar as previsões
tipos = ['CSTR', 'PFR', 'BATCH']
print("\nPrevisões para novos reatores:")
for i, pred in enumerate(saida_novos):
    tipo_previsto = tipos[np.argmax(pred)]
    print(f"Reator {i+1}: tipo previsto = {tipo_previsto} (distribuição: {pred.round(3)})")



Previsões para novos reatores:
Reator 1: tipo previsto = CSTR (distribuição: [0.963 0.037 0.   ])
Reator 2: tipo previsto = PFR (distribuição: [0.375 0.625 0.   ])


**Discussão didática.** Compare as previsões com as classes dos quatro exemplos de treinamento. Observe que dois valores de entrada não são suficientes para definir tecnicamente o tipo de reator de um processo real. Um estudo de aplicação exigiria mais dados, variáveis relevantes e avaliação em um conjunto de teste independente.

**Conclusão didática:** o exercício demonstra a codificação de classes, a normalização das variáveis, a propagação direta, a retropropagação e a classificação por uma rede neural simples. Os resultados ilustram o funcionamento do algoritmo, mas não validam sua utilização para selecionar reatores industriais.

<div style="text-align:center; font-size:12px; color:gray; margin-top:40px; padding-top:15px; border-top:1px solid #dddddd;">
  <div>Material desenvolvido pelo Grupo de Pesquisas em Modelagem Computacional da UESC.</div>
  <div>© 2026 — Todos os direitos reservados. Uso comercial somente mediante autorização.</div>
</div>
